# 使用 1.5 端口 NanoVNA V2 测量 4 端口

NanoVNA V2 是一个 1.5 端口 VNA，这意味着它只能使用两个端口测量正向反射和传输系数 $S_{11}$ 和 $S_{21}$。这被称为三接收器 VNA 架构，尽管有此限制，仍可以进行完全校准的双端口测量。我们建议读者首先阅读 [Calibration With Three Receivers](./Calibration With Three Receivers.ipynb) 以熟悉基本背景。

简而言之，要测量双端口的反向系数 $S_{12}$ 和 $S_{22}$，需要使用端口取向物理翻转（或假翻转）第二次测量该设备，如 [此示例](TwoPortOnePath, EnhancedResponse, and FakeFlip.ipynb)中所述。如果被测设备（DUT）具有更多端口，问题会更严重，如 [此示例](./Measuring a Mutiport Device with a 2-Port Network Analyzer.ipynb)中所述。

在此示例中，使用 NanoVNA V2 通过 Matched Port 技术测量 [4-端口 SMA 功率分配器](https://www.minicircuits.com/WebStore/dashboard.html?model=ZX10Q-2-19-S%2B)。这涉及使用 50 欧姆匹配端接未使用的端口，以所有端口组合测量 4 端口（12 次测量）。对于 VNA 校准，还需要使用 SMA 校准标准进行 4 次更多测量（SHORT、OPEN、MATCH、THROUGH）。DUT 的制造商在其网站上提供测量的 S 参数，这将稍后用于比较。

## 数据采集

## 数据采集

```python
import skrf
from skrf.vi.vna.nanovna import NanoVNAv2

# connect to NanoVNA on /dev/ttyACM0 (Linux)
nanovna = NanoVNAv2('ASRL/dev/ttyACM0::INSTR')

# for Windows users: ASRL1 for COM1
# nanovna = NanoVNAv2('ASRL1::INSTR')

# configure frequency sweep (for example 1 MHz to 4.4 GHz in 1 MHz steps)
f_start = 1e6
f_stop = 4.4e9
f_step = 1e6
num = int(1 + (f_stop - f_start) / f_step)
nanovna.frequency = skrf.Frequency(start=f_start, stop=f_stop, npoints=num)

# measure all 12 combinations of the 4-port
n_ports = 4
for i_src in range(n_ports):
    for i_sink in range(n_ports):
        if i_sink != i_src:
            input('Connect vna_p1 -> dut_p{}, vna_p2 -> dut_p{} and press ENTER:'.format(i_src + 1, i_sink + 1))
            nw_raw = nanovna.get_snp_network(ports=(1, 2))
            nw_raw.write_touchstone('./data_MiniCircuits_splitter/dut_raw_{}{}'.format(i_sink + 1, i_src + 1))
```

校准标准应该使用 `get_snp_network(ports=(1, 2))` 和 `write_touchstone()` 的相同重复调用来测量。

## 使用 `TwoPortOnePath` 进行离线校准

从 NanoVNA 通过 USB 传输的测量数据始终是原始的（未校准的），无论在 NanoVNA 本身上执行了任何校准。这需要使用离线校准校正数据。通过将校准标准的测量存储为独立的 2 端口，可以轻松使用 skrf 创建 TwoPortOnePath 校准。在此示例中，假设测量的 SHORT、OPEN、MATCH 和 THROUGH 的阻抗和相位延迟是理想的，即没有任何损耗或偏移：

```python
import skrf
from skrf.calibration import TwoPortOnePath

# load networks of the raw calibration standard measurements
short_raw = skrf.Network('./data_MiniCircuits_splitter/cal_short_raw.s2p')
open_raw = skrf.Network('./data_MiniCircuits_splitter/cal_open_raw.s2p')
match_raw = skrf.Network('./data_MiniCircuits_splitter/cal_match_raw.s2p')
thru_raw = skrf.Network('./data_MiniCircuits_splitter/cal_thru_raw.s2p')

# create an ideal 50-Ohm line for the short, open, match and through reference responses ("ideals")
line = skrf.media.DefinedGammaZ0(frequency=short_raw.frequency, z0=50)

# create and run the calibration
cal = TwoPortOnePath(ideals=[line.short(nports=2), line.open(nports=2), line.match(nports=2), line.thru()],
                     measured=[short_raw, open_raw, match_raw, thru_raw],
                     n_thrus=1, source_port=1)
cal.run()
```

现在可以使用此校准校正 12 个独立的 2 端口子网络。要进行完全校正，需要成对提供具有正向和反向测量的子网络，例如 ($S_{32}$, $S_{22}$) 与 ($S_{23}$, $S_{33}$) 成对。嵌套循环可以处理此问题。为了与制造商提供的测量进行比较，将校正结果存储在单个 4 端口网络中很方便，然后可以轻松绘制：

```python
import numpy as np

# create an empty array (f x 4 x 4) for the 4-port to be filled
s = np.zeros((len(short_raw.frequency), 4, 4), dtype=complex)
splitter_cal = skrf.Network(frequency=short_raw.frequency, s=s)

# loop through all 12 measurements, apply the calibration and save it inside 4-port network
for i_src in range(4):
    for i_recv in range(4):
        if i_src != i_recv:
            dut_raw_fwd = skrf.Network(f'./data_MiniCircuits_splitter/dut_raw_{i_recv + 1}{i_src + 1}.s2p')
            dut_raw_rev = skrf.Network(f'./data_MiniCircuits_splitter/dut_raw_{i_src + 1}{i_recv + 1}.s2p')
            dut_cal = cal.apply_cal((dut_raw_fwd, dut_raw_rev))

            # dut_cal is now a fully populated and corrected 2-port; save it in splitter_cal
            splitter_cal.s[:, i_src, i_src] = dut_cal.s[:, 0, 0]
            splitter_cal.s[:, i_recv, i_src] = dut_cal.s[:, 1, 0]
            splitter_cal.s[:, i_src, i_recv] = dut_cal.s[:, 0, 1]
            splitter_cal.s[:, i_recv, i_recv] = dut_cal.s[:, 1, 1]
```

结果现在已经校正并组装为单个 4 端口网络。为了比较，将幅度与制造商提供的测量一起绘制：

```python
import matplotlib.pyplot as mplt

# load reference results by MiniCircuits
splitter_mc = skrf.Network('./data_MiniCircuits_splitter/MiniCircuits_ZX10Q-2-19-S+___Plus25degC.s4p')

# plot both results
fig, ax = mplt.subplots(4, 4)
fig.set_size_inches(12, 8)

for i in range(4):
    for j in range(4):
        splitter_cal.plot_s_db(i, j, ax=ax[i][j])
        splitter_mc.plot_s_db(i, j, ax=ax[i][j])
        ax[i][j].get_legend().remove()
        ax[i][j].set_xlim(0, 4.4e9)
fig.legend(['NanoVNA_cal', 'Manufacturer'], loc='upper center', ncol=2)
fig.tight_layout(rect=(0, 0, 1, 0.95))
mplt.show()
```

校正的结果与制造商的非常接近。他们显然使用了一台 Keysight N5242A PNA-X（26.5 GHz 4 端口 VNA）进行测量。考虑到 $200 NanoVNA V2 和 Matched Port 技术的粗糙假设（DUT 未使用端口的端接肯定没有完全消除任何反射），这些结果还不错。